In [ ]:
import os
import re
import cv2
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm

# Scikit-learn Machine Learning components
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, log_loss, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc)


from google.colab import drive
drive.mount('/content/drive')

In [ ]:

KTFE_DATASET_ROOT = Path('/content/drive/MyDrive/my_dataset/thermal')


KTFE_EMOTIONS = ['Anger', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']
ktfe_emotion_classes = {emotion.lower(): idx for idx, emotion in enumerate(KTFE_EMOTIONS)}

def extract_global_thermal_features(img_path, target_size=(64, 64)):

    try:

        img = cv2.imread(str(img_path))
        if img is None:
            return None
        img = cv2.resize(img, target_size)


        gray_image = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)


        gray_normalized = gray_image.astype(np.float32) / 255.0


        processed_canvas = cv2.GaussianBlur(gray_normalized, (5, 5), 0)


        return processed_canvas.flatten()

    except Exception as e:
        print(f"Error processing image {img_path}: {e}")
        return None

In [ ]:
def load_global_ktfe_dataset(root_path):

    X_list, y_list, paths_list = [], [], []
    all_extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
    all_files = []

    for ext in all_extensions:
        all_files.extend(list(root_path.rglob(ext)))
    all_files = sorted(list(set(all_files)))

    print(f"Discovered {len(all_files)} files inside the target KTFE Drive folder.")
    print("Compiling global flattened feature matrices...")

    for img_path in tqdm(all_files, desc="Processing KTFE Files"):
        filename_lower = img_path.name.lower()
        assigned_label = None


        for emotion in KTFE_EMOTIONS:
            if emotion.lower() in filename_lower:
                assigned_label = ktfe_emotion_classes[emotion.lower()]
                break

        if assigned_label is None:
            continue

        features = extract_global_thermal_features(img_path)
        if features is not None:
            X_list.append(features)
            y_list.append(assigned_label)
            paths_list.append(str(img_path))

    return np.array(X_list), np.array(y_list), np.array(paths_list)

print("STARTING GLOBAL FULL-FRAME EXTRACTION LOOP FOR KTFE (7 CLASSES)...")
X_ktfe, y_ktfe, processed_ktfe_paths = load_global_ktfe_dataset(KTFE_DATASET_ROOT)

print("\nGLOBAL EXTRACTION COMPLETE WORKSPACE SUMMARY")
print(f"Total Extracted KTFE Samples (Rows): {len(X_ktfe)}")
print(f"Total Matrix Feature Columns: {X_ktfe.shape[1]} (Flattened Global Pixels)")

In [ ]:
print("Performing Stratified Frame-Level Holdout Split...")


X_train, X_test, y_train, y_test = train_test_split(
    X_ktfe, y_ktfe, test_size=0.2, stratify=y_ktfe, random_state=42
)

print("Fitting standard scale metrics strictly on the training partition...")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)


X_test_scaled = scaler.transform(X_test)

print("Data scaling completed safely!")
print(f" -> Training Set Volume: {len(X_train_scaled)} frames.")
print(f" -> Testing Set Volume:  {len(X_test_scaled)} frames.")

In [ ]:
print("Training final non-linear Global KTFE SVM model...")

svm_classifier = SVC(
    kernel='rbf',
    C=10.0,
    gamma='scale',
    class_weight='balanced',
    probability=True,
    cache_size=2000,
    random_state=42
)


for _ in tqdm(range(1), desc="Model Fitting"):
    svm_classifier.fit(X_train_scaled, y_train)

print("Global SVM Model training completed safely.")

In [ ]:
print("Running test matrix predictions...")
y_pred = svm_classifier.predict(X_test_scaled)
y_prob = svm_classifier.predict_proba(X_test_scaled)

print("\n================ GLOBAL PERFORMANCE METRICS ================")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f} %")
print(f"Log Loss Value: {log_loss(y_test, y_prob):.4f}")

print("\nClassification Report (KTFE Global Features):")
unique_test_classes = np.unique(y_test)
present_labels = [KTFE_EMOTIONS[idx] for idx in unique_test_classes]
print(classification_report(y_test, y_pred, labels=unique_test_classes, target_names=present_labels))


cm = confusion_matrix(y_test, y_pred, labels=unique_test_classes)
plt.figure(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=present_labels)
disp.plot(cmap='Blues', ax=plt.gca())
plt.title("KTFE Global Full-Frame Confusion Matrix", fontsize=12, fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
print("Generating dynamically indexed class-wise ROC curve plots...")

n_classes = len(KTFE_EMOTIONS)
y_test_binarized = label_binarize(y_test, classes=np.arange(n_classes))
plot_colors = ['#E41A1C', '#377EB8', '#4DAF4A', '#984EA3', '#FF7F00', '#FFD700', '#000000']

plt.figure(figsize=(15, 10))

for i in range(n_classes):
    if np.sum(y_test_binarized[:, i]) == 0:
        print(f"Skipping ROC plot for {KTFE_EMOTIONS[i]}: No samples present in this test subset.")
        continue

    fpr, tpr, _ = roc_curve(y_test_binarized[:, i], y_prob[:, i])
    roc_auc = auc(fpr, tpr)


    plt.subplot(3, 3, i + 1)
    plt.plot(fpr, tpr, color=plot_colors[i % len(plot_colors)], linewidth=2.5, label=f'AUC = {roc_auc:.4f}')
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1.5, alpha=0.5)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=9)
    plt.ylabel('True Positive Rate', fontsize=9)
    plt.title(f'ROC Curve: {KTFE_EMOTIONS[i]}', fontsize=11, fontweight='bold')
    plt.legend(loc="lower right", frameon=True)
    plt.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()